### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [ ]:
import os
import sys
from langchain_community.document_loaders import ConfluenceLoader
from atlassian import Confluence

sys.path.append(os.path.abspath(".."))

In [ ]:
# Define constants in constants.py and import them here
from data.constants import token, username, url, url_extended, embedding_model_name

In [ ]:

# Initialize Confluence client
confluence = Confluence(
    url=url,
    username=username,
    password=token,
    cloud=True
)

# Get all spaces from Confluence
spaces = confluence.get_all_spaces(start=0, limit=100)
confluence_documents = []

# Loop through spaces
for space in spaces.get("results", []):
    space_key = space["key"]
    print(f"Loading space: {space_key}")

    # Initialize ConfluenceLoader for the current space
    loader = ConfluenceLoader(
        url=url_extended,
        username=username,
        api_key=token,
        space_key=space_key
    )

    # Load documents from the current space
    docs = loader.load()
    
    # Append loaded documents to the main list
    confluence_documents.extend(docs)

print(f"Total docs loaded: {len(confluence_documents)}")


In [ ]:
print(confluence_documents)

In [ ]:
from src.utility import split_documents

chunks=split_documents(confluence_documents)
chunks

In [ ]:
from src.embedding_manager import EmbeddingManager

embedding_manager=EmbeddingManager(embedding_model_name)
embedding_manager

In [ ]:
### embedding And vectorStoreDB
from src.vector_store import VectorStore
from data.constants import vector_db_name


vectorstore=VectorStore(collection_name=vector_db_name)
vectorstore


In [ ]:
chunks

In [ ]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

In [ ]:
### Retriever Pipeline From VectorStore

In [ ]:
from src.rag_retriever import RAGRetriever

rag_retriever=RAGRetriever(vectorstore,embedding_manager)
rag_retriever


In [ ]:
rag_retriever.retrieve("Lcqmc: A large-scale chinese question matching corpus")

In [ ]:
from src.grok_llm import GroqLLM
from data.constants import groq_api_key

# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=groq_api_key)
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

    

In [ ]:
rag_retriever.retrieve("Lcqmc: A large-scale chinese question matching corpus")

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from src.rag_pipeline import RAGPipeline
from langchain_groq import ChatGroq

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

# Example usage:
adv_rag = RAGPipeline(rag_retriever, llm)


In [ ]:
answer=adv_rag.query_simple("Lcqmc: A large-scale chinese question matching corpus")
print(answer)

In [ ]:

result = adv_rag.query_advanced("Lcqmc: A large-scale chinese question matching corpus", top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

In [ ]:

result = adv_rag.query_formatted("Lcqmc: A large-scale chinese question matching corpus", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])